In [1]:
import Exam as exam
import numpy as np
import pandas as pd
from scipy.stats import t, norm, multivariate_normal, skew, kurtosis
import statsmodels.miscmodels.tmodel as tmodel
from statsmodels.stats.correlation_tools import cov_nearest, corr_nearest

2. (20 pts) Using problem2.csv </br>
a. Calculate the Mean, Variance, Skewness and Kurtosis of the data (8)

In [2]:
# load data
data2 = np.loadtxt('problem2.csv', skiprows=1)

In [3]:
mean = np.mean(data2)
var = np.var(data2)
skewness = skew(data2)
kurt = kurtosis(data2)
print(mean, var, skewness, kurt)

-0.00034577495508131593 0.00048493780039086763 0.11408120646692728 0.9575595460250157


b. Given a choice between a normal distribution and a t-distribution, which one would
you choose to model the data and why based on part a alone (4)? </br>
I would choose the T distribution because Kurtosis is larger than 0.

c. Fit both distributions and prove or disprove your choice in b. (8)

In [4]:
# normal 
est_mu = np.mean(data2)
est_sigma = np.std(data2, ddof=1)
print(est_mu, est_sigma)

-0.00034577495508131593 0.02203232224742735


In [5]:
# T
est_nu, est_mu, est_sigma = t.fit(data2)
print(est_nu, est_mu, est_sigma)

8.85936027362565 -0.0004775894210559369 0.019398966622270863


In [6]:
def model_selection_metrics(data):
    n = len(data)

    # Normal
    mu_n = np.mean(data)
    sigma_n = np.std(data, ddof=1)
    loglik_n = np.sum(norm.logpdf(data, mu_n, sigma_n))
    k_n = 2

    AIC_n = 2*k_n - 2*loglik_n
    BIC_n = k_n*np.log(n) - 2*loglik_n
    AICc_n = AIC_n + (2*k_n*(k_n+1)) / (n - k_n - 1)

    # T 
    nu_t, mu_t, sigma_t = t.fit(data)
    loglik_t = np.sum(t.logpdf(data, nu_t, mu_t, sigma_t))
    k_t = 3

    AIC_t = 2*k_t - 2*loglik_t
    BIC_t = k_t*np.log(n) - 2*loglik_t
    AICc_t = AIC_t + (2*k_t*(k_t+1)) / (n - k_t - 1)

    return AIC_n, BIC_n, AICc_n, AIC_t, BIC_t, AICc_t

In [7]:
AIC_n, BIC_n, AICc_n, AIC_t, BIC_t, AICc_t = model_selection_metrics(data2)
AICc_n, AICc_t

(np.float64(-4789.600319016607), np.float64(-4809.55817385082))

T distribution is better with a lower AICc

3. (20 pts) Using problem2.csv and your fitted models. These are returns of a stock </br>
a. Calculate the VaR (5% alpha) as distance from 0 for both models (8) </br>
b. Calculate the ES (5% alpha) as the distance from 0 for both models (8)

In [8]:
# normal
var_abs, var_diff, es_abs, es_diff = exam.var_es_normal(data2)
print(var_abs, es_abs)

0.036585720113925815 0.045792128233980496


In [9]:
# T
var_abs, var_diff, es_abs, es_diff = exam.var_es_t(data2)
print(var_abs, es_abs)

0.03610246447166382 0.04823023095412436


c. Discuss the results. What do you notice? Why is that? (4) </br>
The VaR values are similar between the two models, with the normal distribution having a slightly higher VaR. However, the ES values show a more pronounced difference, with the t-distribution yielding a higher ES.
This makes sense.  The t-distribution has more kurtosis (heavier tails) than the normal distribution, which means that it predicts more extreme losses in the tail of the distribution.  Therefore, while the 95% tile is lower for the t-distribution, the ES values reflect the increased risk of extreme losses captured by the t-distribution.

4. (10 pts) Using problem4.csv </br>
a. Calculate the exponentially weighted correlation matrix with lambda = 0.94 (3)

In [10]:
data4 = pd.read_csv('problem4.csv')
data4

,x1,x2,x3
0,0.034246,0.121262,-0.115528
1,0.131169,0.192027,0.043622
2,0.125515,-0.108881,-0.004787
3,0.157331,0.033694,0.109034
4,-0.119847,-0.249491,-0.234808
...,...,...,...
495,-0.087194,-0.062952,-0.125135
496,-0.015536,-0.134150,0.012078
497,-0.182251,-0.144493,-0.240579
498,0.167292,0.207168,0.128010


In [11]:
ew_corr_last = exam.ew_corr(data4, 0.94)
ew_corr_last

,x1,x2,x3
x1,1.000000,0.711329,0.807175
x2,0.711329,1.000000,0.713020
x3,0.807175,0.713020,1.000000


b. Calculate the exponentially weighted variances with lambda=0.97 (3)

In [12]:
ew_cov_biased, var_biased = exam.ew_cov_var(data4, 0.97)
var_biased

x1    0.015379
x2    0.035517
x3    0.027813
Name: 499, dtype: float64

c. Combine A and B to form a covariance matrix (3)

In [13]:
ew_sd = np.sqrt(var_biased)
D = np.diag(ew_sd)

cov = D @ ew_corr_last @ D

cov

,0,1,2
0,0.015379,0.016625,0.016694
1,0.016625,0.035517,0.022410
2,0.016694,0.022410,0.027813


d. Why would you do something like this in practice? (1) </br>
In practice, we would mix weightings when we think the updating of correlations 
and volatilies are different.  Here we think the correlations update faster (lower lambda)
than the variances (higher lambda).  The higher the lambda, the more persistance 
of older data.  The lower the lambda, the more weight is given to more recent data.

5. (30 pts) Using the data in problem5.csv. These data contain missing values </br>
a. Calculate the pairwise covariance of the data (10)

In [77]:
data5 = pd.read_csv('problem5.csv')
data5

,x1,x2,x3,x4,x5
0,0.378172,0.508313,NaN,0.610538,NaN
1,0.858478,0.758142,NaN,NaN,NaN
2,-0.218954,-0.015334,-0.261880,NaN,-0.230148
3,-0.370442,-0.424702,-0.535312,NaN,-0.226233
4,0.717732,0.664195,NaN,NaN,0.914107
5,1.331701,NaN,1.464934,1.156675,1.397827
6,1.462355,NaN,1.396949,1.476854,NaN
7,0.560968,0.570253,0.622116,0.570309,0.417396
8,NaN,1.343589,1.578797,1.576128,NaN
9,NaN,NaN,-2.841926,-2.845942,NaN


In [78]:
# Pairwise cov
cov = data5.cov()
cov

,x1,x2,x3,x4,x5
x1,1.470484,1.454214,0.877269,1.903226,1.444361
x2,1.454214,1.252078,0.539548,1.621918,1.237877
x3,0.877269,0.539548,1.272425,1.171959,1.091912
x4,1.903226,1.621918,1.171959,1.814469,1.589729
x5,1.444361,1.237877,1.091912,1.589729,1.396186


b. Is your matrix Positive Definite, Positive Semi-definite, or Non Definite? (5)

In [20]:
exam.eigenvalue(cov)

Indefinite


c. If the matrix is non definite, use Higham’s method to fix the matrix. (10)

In [81]:
std = np.sqrt(np.diag(cov))
corr = cov / np.outer(std, std)
higham_corr = corr_nearest(corr)
D = np.diag(std)
higham_cov = D @ higham_corr @ D

pd.DataFrame(higham_cov)

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/stats/correlation_tools.py:89: IterationLimitWarning: 
Maximum iteration reached.

  warnings.warn(iteration_limit_doc, IterationLimitWarning)


,0,1,2,3,4
0,1.470484,1.332361,0.884378,1.627602,1.399556
1,1.332361,1.252078,0.619028,1.450604,1.214450
2,0.884378,0.619028,1.272425,1.076846,1.059658
3,1.627602,1.450604,1.076846,1.814469,1.577928
4,1.399556,1.214450,1.059658,1.577928,1.396186


d. For each principal component, list the variance explained and the cumulative
variance explained, sorted from largest to smallest variance explained. (5)

In [24]:
# eigen decompose
eigen_val, eigen_vec = np.linalg.eigh(cov)
    
# sort descending
idx = np.argsort(eigen_val[::-1])
eigen_val = eigen_val[idx]
eigen_vec = eigen_vec[:, idx]
    
# variance explained
total_var = eigen_val.sum()
pct_explained = eigen_val / total_var
cum_explained = np.cumsum(pct_explained)

In [31]:
output = pd.DataFrame(columns=['Explained', 'Cumulative'])
output['Explained'] = pct_explained
output['Cumulative'] = cum_explained
output

,Explained,Cumulative
0,0.941860,0.941860
1,0.115803,1.057663
2,0.003883,1.061545
3,-0.018490,1.043056
4,-0.043056,1.000000


6. (20 pts) Using problem6.csv. These data are prices of 3 stocks. You own 100 shares of each
stock. Using arithmetic returns: </br>
a. De-mean the return series so that the mean of each is 0. Fit a Student T model for
each stock. Report the fit values. (3)

In [62]:
data6 = pd.read_csv('problem6.csv')
data6

,x1,x2,x3,Date
0,102.975983,101.234230,101.390965,2025-01-01
1,102.553769,102.056711,103.589863,2025-01-02
2,102.362551,100.605815,99.204933,2025-01-03
3,104.522707,100.479907,101.798573,2025-01-04
4,103.682209,100.637430,101.599604,2025-01-05
...,...,...,...,...
245,79.313328,74.647039,78.285858,2025-09-03
246,78.871520,74.591839,77.059249,2025-09-04
247,80.080646,74.965529,77.739804,2025-09-05
248,81.411350,75.702936,79.729444,2025-09-06


In [39]:
returns = exam.arith_return(data6)
returns

,x1,x2,x3
Date,,,
2025-01-02,-0.004100,0.008125,0.021687
2025-01-03,-0.001865,-0.014217,-0.042330
2025-01-04,0.021103,-0.001252,0.026144
2025-01-05,-0.008041,0.001568,-0.001955
2025-01-06,-0.003674,-0.006858,-0.027222
...,...,...,...
2025-09-03,-0.017321,-0.029331,-0.002462
2025-09-04,-0.005570,-0.000739,-0.015668
2025-09-05,0.015330,0.005010,0.008832


In [41]:
demean = returns - returns.mean()
demean.mean()

x1   -8.638784e-19
x2    1.476953e-18
x3   -3.148976e-18
dtype: float64

In [48]:
x1_return = demean['x1'].values
x2_return = demean['x2'].values
x3_return = demean['x3'].values

x1_nu, x1_mu, x1_sigma = t.fit(x1_return)
x2_nu, x2_mu, x2_sigma = t.fit(x2_return)
x3_nu, x3_mu, x3_sigma = t.fit(x3_return)

print(x1_nu, x1_mu, x1_sigma)
print(x2_nu, x2_mu, x2_sigma)
print(x3_nu, x3_mu, x3_sigma)

4.729830909131829 -0.0004786520929827172 0.012907549628639717
6.766945042505089 -4.350220543514708e-05 0.009058419601620012
39.864383658541556 7.48934264331786e-05 0.01706272432180566


b. Simulate the system using a Gaussian Copula. Report the correlation matrix you
used in the copula. (3)

In [51]:
# Portfolio VaR & ES on T dist, from Copula
# transform to uniforms
U_x1 = t.cdf(x1_return, df=x1_nu, loc=x1_mu, scale=x1_sigma)
U_x2 = t.cdf(x2_return, df=x2_nu, loc=x2_mu, scale=x2_sigma)
U_x3 = t.cdf(x3_return, df=x3_nu, loc=x3_mu, scale=x3_sigma)
U = np.column_stack([U_x1, U_x2, U_x3])

# Gaussian Copula corr
Z = norm.ppf(np.column_stack([U_x1, U_x2, U_x3]))
copula_corr = np.corrcoef(Z.T)

copula_corr

array([[1.        , 0.4648933 , 0.43759569],
       [0.4648933 , 1.        , 0.51901222],
       [0.43759569, 0.51901222, 1.        ]])

c. What is the VaR and ES at the 5% alpha level for each stock expressed in $? (7)

In [65]:
# simulate joint uniforms
np.random.seed(42)
n_sims = 100_000
sim = multivariate_normal.rvs(np.zeros(3), copula_corr, n_sims)
U_sim = norm.cdf(sim)

# transform back to T dist
x1_sim = t.ppf(U_sim[:, 1], df=x1_nu, loc=x1_mu, scale=x1_sigma)
x2_sim = t.ppf(U_sim[:, 1], df=x2_nu, loc=x2_mu, scale=x2_sigma)
x3_sim = t.ppf(U_sim[:, 1], df=x3_nu, loc=x3_mu, scale=x3_sigma)
    
R_sim = np.column_stack([x1_sim, x2_sim, x3_sim])

# convert return to P&L
x1_pos_value = 100 * data6.loc[249, "x1"]
x2_pos_value = 100 * data6.loc[249, "x2"]
x3_pos_value = 100 * data6.loc[249, "x3"]
V0 = x1_pos_value + x2_pos_value + x3_pos_value
    
# weighted portfolio return
x1_weight = x1_pos_value / V0
x2_weight = x2_pos_value / V0
x3_weight = x3_pos_value / V0
    
weights = np.array([x1_weight, x2_weight, x3_weight])
    
x1_loss = -x1_sim * x1_pos_value
x2_loss = -x2_sim * x2_pos_value
x3_loss = -x3_sim * x3_pos_value
    
R_port = x1_weight * x1_sim + x2_weight * x2_sim + x2_weight * x2_sim

In [72]:
alpha = 0.05

def var_es(sim, pos_value, nu, mu, sigma):
    var_pct = abs(np.quantile(sim, alpha))
    var = var_pct * pos_value
        
    t_stat = t.ppf(alpha, df=nu)
    phi_t = t.pdf(t_stat, df=nu)
    es_pct = -mu + sigma * ((nu + t_stat**2) / (nu - 1)) * (phi_t / alpha)
    es = es_pct * pos_value
    return var, es, var_pct, es_pct

In [75]:
x1_var, x1_es, x1_var_pct, x1_es_pct = var_es(x1_sim, x1_pos_value, x1_nu, x1_mu, x1_sigma)
x2_var, x2_es, x2_var_pct, x2_es_pct = var_es(x2_sim, x2_pos_value, x2_nu, x1_mu, x2_sigma)
x3_var, x3_es, x3_var_pct, x3_es_pct = var_es(x3_sim, x3_pos_value, x3_nu, x3_mu, x3_sigma)

print(x1_var, x1_es, x1_var_pct, x1_es_pct)
print(x2_var, x2_es, x2_var_pct, x2_es_pct)
print(x3_var, x3_es, x3_var_pct, x3_es_pct)

221.3075302115643 320.97241433007815 0.026643266091912716 0.03864194514748176
133.05058635068508 187.26927517474903 0.017188918622846185 0.024193477231688847
234.2241927505326 299.2929917724518 0.02850963671417815 0.036429774253166715


d. What is the VaR and ES at the 5% alpha level for the total portfolio expressed in $?
(7)

In [76]:
q_alpha = np.quantile(R_port, alpha)
var_pct = -q_alpha
es_pct = -R_port[R_port <= q_alpha].mean()
var = V0 * var_pct
es = V0 * es_pct
print(var, es, var_pct, es_pct)

487.4087029129345 689.9875899821536 0.020089037167418552 0.02843852860519204
